# SKU MatchOps — Colab GPU Classifier & ArcFace Training

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Set your `GOOGLE_SHEET_ID` and `DOMAIN` (`food`, `market`, or `all`) in Cell 2.
3. Run all cells — at the end it downloads `matchops_colab_artifacts.zip`.
4. Extract `matchops_colab_artifacts.zip` into your local `sku-matchops` repo root, then run locally:
   ```bash
   python -m scripts.sync --qdrant sync
   ```

In [ ]:
# @title 1. Check GPU, Clone Repo & Install Dependencies
!nvidia-smi

REPO_URL = "https://github.com/muaazl/sku-matchops.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

from pathlib import Path

if not Path("sku-matchops").exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
%cd sku-matchops
!git pull --ff-only

!pip install -q "scikit-learn==1.9.1" onnxscript onnxruntime polars pyarrow joblib quantulum3 rapidfuzz python-dotenv rich tqdm transformers huggingface_hub openpyxl

In [ ]:
# @title 2. Train Classifiers + ArcFace on GPU
GOOGLE_SHEET_ID = ""  # @param {type:"string"}
DOMAIN = "food"  # @param ["food", "market", "all"]
EPOCHS = 25  # @param {type:"integer"}
EMBED_BATCH_SIZE = 256  # @param {type:"integer"}
EMBEDDING_MODEL = "jinaai/jina-embeddings-v5-text-small"  # @param {type:"string"}

assert GOOGLE_SHEET_ID.strip(), "Please enter your GOOGLE_SHEET_ID above!"

!python -m scripts.ops.colab_train_classifiers \
  --sheet-id "{GOOGLE_SHEET_ID.strip()}" \
  --domain "{DOMAIN}" \
  --epochs {EPOCHS} \
  --embed-batch-size {EMBED_BATCH_SIZE} \
  --embedding-model "{EMBEDDING_MODEL}" \
  --output-zip matchops_colab_artifacts.zip

In [ ]:
# @title 3. Download matchops_colab_artifacts.zip
from google.colab import files

files.download("matchops_colab_artifacts.zip")